# 1.语义分割器使用示例

知识点讲解：语义文档分割器（SemanticChunker）

1. 组件定位：语义分割器（SemanticChunker）解决什么问题
   - 传统字符分割器（如 RecursiveCharacterTextSplitter）只按字符数 / 分隔符等表面特征切分，不关心句子间的语义关联
   - SemanticChunker 基于文本语义相似度切分，能智能识别主题边界，让每个块尽量保持语义连贯
   - 适用场景：长文本需要切成「语义相关」的块、问答 / 知识库等对语义完整性要求高的任务

2. 实验性定位与包归属（重要）
   - 该类目前位于 langchain_experimental 包，安装命令：pip install -Uqq langchain_experimental
   - 实验性意味着类与方法未来大概率会发生变更，需谨慎用于生产
   - 与多数分割器不同，SemanticChunker 并没有继承 TextSplitter，因此 API 形态略有差异

3. 工作原理（五步）
   - 第一步：按 sentence_split_regex 把文档拆成独立的句子
   - 第二步：根据 buffer_size（默认 1）前后拼接相邻句子，形成待编码的小段
   - 第三步：用传入的 Embedding 模型对每段计算向量
   - 第四步：按传入的「分块数 + 断点类型」计算一个相似度阈值
   - 第五步：把相似度高于阈值的相邻段合并，低于阈值的作为切块边界

4. 核心参数详解
   - embeddings：文本嵌入模型，底层用向量余弦相似度识别句子相似性（必填）
   - buffer_size：缓冲区大小，默认 1，计算相似度时该句叠加前后各 1 条文本，首尾不足则不叠加
   - add_start_index：是否添加起点索引，默认 False（本例设为 True 便于定位）
   - breakpoint_threshold_type：断点阈值类型，默认 percentile（百分位）
   - breakpoint_threshold_amount：断点阈值的具体得分 / 金额
   - number_of_chunks：期望生成的块数量，默认 None（本例设为 10）
   - sentence_split_regex：句子切割正则，默认 (?<=[.?!])\s+；本例用 r"(?<=[。？！.?!])" 适配中英文句末标点

5. 底层断点检测算法
   - 目前支持 4 种阈值判定方法：百分位（默认）、标准差、四分位数、梯度
   - 不同方法会影响切块粒度，百分位适合大多数分布较均匀的文本

6. 典型输出示例与观察点
   - 按本例参数切 10 块后输出类似：
       块大小: 201, 元数据: {'source': './科幻短篇.txt'}
       块大小: 25,  元数据: {'source': './科幻短篇.txt'}
       块大小: 31,  元数据: {'source': './科幻短篇.txt'}
       块大小: 466, 元数据: {'source': './科幻短篇.txt'}
       块大小: 0,   元数据: {'source': './科幻短篇.txt'}
   - 观察点 1：各块大小差异很大，因为切块由语义断点而非固定字符数决定
   - 观察点 2：可能出现大小为 0 的空块（语义断点恰好落在句边界），生产可过滤

7. 同类语义分割器补充
   - 除基于 Embedding 的 SemanticChunker，LangChain 还提供 3 种基于 NLP 的语义分割器：
     * NLTKTextSplitter：基于 NLTK 自然语言处理库
     * SpacyTextSplitter：基于 spaCy 高级 NLP 库
     * SentenceTransformersTokenTextSplitter：按句子转换器模型的 token 窗口切分

8. 优势、局限与注意事项
   - 优势：保持主题完整性、适合高语义连贯性场景
   - 局限：要调用 Embedding API 成本高、速度慢、依赖 Embedding 模型质量、大文档产生大量 API 调用
   - 注意事项：需要网络调用 Embedding API；建议配合 langchain_experimental 使用；生产环境注意空块过滤与 API 限流


In [ ]:
import dotenv
from langchain_unstructured import UnstructuredLoader
from langchain_experimental.text_splitter import SemanticChunker
from langchain_openai import OpenAIEmbeddings
dotenv.load_dotenv()
loader = UnstructuredLoader("./科幻短篇.txt")
text_splitter = SemanticChunker(
    embeddings=OpenAIEmbeddings(model="text-embedding-3-small"),
    number_of_chunks=10,
    add_start_index=True,
    sentence_split_regex=r"(?<=[。？！.?!])"
)
documents = loader.load()
chunks = text_splitter.split_documents(documents)
for chunk in chunks:
    print(f"块大小: {len(chunk.page_content)}, 元数据: {chunk.metadata}")